# Monocular depth estimation with a Vision Transformer

Train a compact ViT from scratch on paired KITTI images and depth maps. The model uses 16×16 image patches, learned position embeddings, self-attention and a convolutional depth head that upsamples to 256×512. It uses all patch tokens for dense prediction.

Run top to bottom. In Colab, select a GPU runtime first. Set `DATA_ROOT` to your extracted dataset, or follow the download instructions below.

In [ ]:
from pathlib import Path
import math
import re

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit

DATA_ROOT = Path("data")
CHECKPOINT = Path("vit_depth_best.pt")
HEIGHT, WIDTH = 256, 512
SEED = 42
MAX_EPOCHS, PATIENCE = 40, 5
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Data setup

In Colab, run the next cell to download and extract KITTI if the depth files are missing. Configure Kaggle CLI credentials first if prompted. In local Jupyter, skip that cell and set DATA_ROOT in the first code cell to the extracted dataset.

The loader matches left-camera frames by drive and frame ID and holds out whole drives for internal validation.

In [ ]:
from pathlib import Path
import subprocess
import sys

DATA_ROOT = Path("/content/data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
if not any(DATA_ROOT.glob("**/proj_depth/groundtruth/image_02/*.png")):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    subprocess.run([
        "kaggle", "datasets", "download", "-d",
        "aalniak/kitti-eigen-split-for-monocular-depth-estimation",
        "--unzip", "-p", str(DATA_ROOT),
    ], check=True)

rgb_count = sum(
    p.suffix.lower() in {".png", ".jpg", ".jpeg"}
    for p in DATA_ROOT.glob("**/image_02/data/*")
)
depth_count = sum(1 for _ in DATA_ROOT.glob("**/proj_depth/groundtruth/image_02/*.png"))
print(f"RGB files: {rgb_count} | Depth files: {depth_count}")
if not rgb_count or not depth_count:
    raise FileNotFoundError(f"Expected both RGB and depth files under {DATA_ROOT}.")

In [ ]:
def find_pairs(root):
    drive_pattern = re.compile(r"\d{4}_\d{2}_\d{2}_drive_\d+_sync")

    def index_frames(paths, prefer_png=False):
        frames = {}
        for path in sorted(paths):
            drive = next((p.name for p in path.parents if drive_pattern.fullmatch(p.name)), None)
            if drive is None:
                continue
            key = (drive, path.stem)
            previous = frames.get(key)
            if previous is None:
                frames[key] = path
            elif prefer_png and previous.parent == path.parent and previous.suffix.lower() != path.suffix.lower():
                if path.suffix.lower() == ".png":
                    frames[key] = path
            else:
                raise ValueError(f"Duplicate frame {key}: {previous} and {path}")
        return frames

    rgb = index_frames(
        (p for p in root.glob("**/image_02/data/*")
         if p.suffix.lower() in {".png", ".jpg", ".jpeg"}),
        prefer_png=True,
    )
    depth = index_frames(root.glob("**/proj_depth/groundtruth/image_02/*.png"))
    return [(rgb[key], depth[key], key[0]) for key in sorted(rgb.keys() & depth.keys())]


def read_rgb(path):
    with Image.open(path) as image:
        pixels = np.asarray(
            image.convert("RGB").resize((WIDTH, HEIGHT), Image.Resampling.BILINEAR),
            dtype=np.float32,
        )
    return torch.from_numpy(pixels).permute(2, 0, 1) / 255.0


class KITTIDepth(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        rgb_path, depth_path, _ = self.pairs[index]
        with Image.open(depth_path) as depth:
            depth = depth.resize((WIDTH, HEIGHT), Image.Resampling.NEAREST)
            meters = np.asarray(depth, dtype=np.float32) / 256.0
        return read_rgb(rgb_path), torch.from_numpy(meters).unsqueeze(0)


pairs = find_pairs(DATA_ROOT)
drives = [drive for _, _, drive in pairs]
if len(set(drives)) < 2:
    raise ValueError(f"Found {len(pairs)} pairs in {DATA_ROOT.resolve()}. Extract KITTI with at least two paired drives.")
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(splitter.split(pairs, groups=drives))
train_ds = KITTIDepth([pairs[i] for i in train_idx])
val_ds = KITTIDepth([pairs[i] for i in val_idx])
print(f"Matched {len(pairs)} pairs | Train: {len(train_ds)} | Validation: {len(val_ds)}")
print(f"Validation drives: {sorted({drives[i] for i in val_idx})}")

batch_size = 4 if device.type == "cuda" else 1
loader_options = dict(batch_size=batch_size, pin_memory=device.type == "cuda")
train_loader = DataLoader(train_ds, shuffle=True, **loader_options)
val_loader = DataLoader(val_ds, **loader_options)

## Vision Transformer and depth head

The image becomes a 16×32 grid of patch tokens. Four Transformer encoder blocks let each patch attend to other patches. The depth head reshapes the tokens back into a grid and upsamples it to the input resolution. Softplus produces positive depths in metres.

In [ ]:
class ViTDepth(nn.Module):
    def __init__(self):
        super().__init__()
        patch_size, dim = 16, 128
        self.grid = (HEIGHT // patch_size, WIDTH // patch_size)
        self.patch_embedding = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size)
        self.position = nn.Parameter(torch.empty(1, math.prod(self.grid), dim))
        nn.init.trunc_normal_(self.position, std=0.02)
        block = nn.TransformerEncoderLayer(
            dim, nhead=4, dim_feedforward=4 * dim, dropout=0.1,
            activation="gelu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            block, num_layers=4, norm=nn.LayerNorm(dim), enable_nested_tensor=False,
        )
        self.depth_head = nn.Sequential(
            nn.Conv2d(dim, 64, 3, padding=1),
            nn.GELU(),
            nn.Upsample(scale_factor=4, mode="bilinear", align_corners=False),
            nn.Conv2d(64, 32, 3, padding=1),
            nn.GELU(),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(32, 1, 3, padding=1),
        )

    def forward(self, rgb):
        patches = self.patch_embedding(rgb)
        tokens = patches.flatten(2).transpose(1, 2)
        features = self.encoder(tokens + self.position).transpose(1, 2)
        features = features.reshape(rgb.shape[0], -1, *self.grid)
        depth = self.depth_head(features)
        return F.softplus(F.interpolate(
            depth, size=rgb.shape[-2:], mode="bilinear", align_corners=False,
        ))


model = ViTDepth().to(device)
with torch.no_grad():
    example = model.eval()(torch.zeros(1, 3, HEIGHT, WIDTH, device=device))
assert example.shape == (1, 1, HEIGHT, WIDTH)
assert torch.isfinite(example).all() and (example > 0).all()
print(f"Output shape: {tuple(example.shape)}")

## Training and early stopping

Training minimizes mean absolute error on pixels with a LiDAR depth measurement. Validation MAE and RMSE are aggregated over valid pixels. Training stops after five consecutive epochs without a lower validation MAE or at 40 epochs. Inference uses the checkpoint from the lowest validation MAE.

In [ ]:
# Rebuild loaders if the datasets exist but the loader setup was skipped.
if "train_loader" not in globals() or "val_loader" not in globals():
    if "train_ds" not in globals() or "val_ds" not in globals():
        raise RuntimeError(
            "Run the KITTI download and dataset pairing cells first. "
            "Wait for the 'Matched ... pairs' output before training."
        )
    batch_size = 4 if device.type == "cuda" else 1
    loader_options = dict(batch_size=batch_size, pin_memory=device.type == "cuda")
    train_loader = DataLoader(train_ds, shuffle=True, **loader_options)
    val_loader = DataLoader(val_ds, **loader_options)

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    abs_error = squared_error = 0.0
    valid_pixels = 0
    for rgb, target in loader:
        valid = target > 0
        count = int(valid.sum())
        if count == 0:
            continue
        rgb, target, valid = rgb.to(device), target.to(device), valid.to(device)
        with torch.set_grad_enabled(training):
            error = model(rgb)[valid] - target[valid]
            absolute = error.abs().sum()
            loss = absolute / count
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
        abs_error += absolute.detach().item()
        squared_error += error.detach().square().sum().item()
        valid_pixels += count
    if valid_pixels == 0:
        raise ValueError("This split has no valid depth pixels.")
    return abs_error / valid_pixels, math.sqrt(squared_error / valid_pixels)


optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
best_mae, best_epoch, stale_epochs = float("inf"), 0, 0
for epoch in range(1, MAX_EPOCHS + 1):
    train_mae, _ = run_epoch(model, train_loader, optimizer)
    val_mae, val_rmse = run_epoch(model, val_loader)
    if not all(map(math.isfinite, (train_mae, val_mae, val_rmse))):
        raise FloatingPointError(f"Non-finite metric at epoch {epoch}.")
    print(f"Epoch {epoch:02d} | Train MAE {train_mae:.3f} m | "
          f"Val MAE {val_mae:.3f} m | Val RMSE {val_rmse:.3f} m")
    if val_mae < best_mae:
        best_mae, best_epoch, stale_epochs = val_mae, epoch, 0
        torch.save(model.state_dict(), CHECKPOINT)
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            print(f"Early stopping at epoch {epoch}.")
            break

model.load_state_dict(torch.load(CHECKPOINT, map_location=device, weights_only=True))
model.eval()
print(f"Restored epoch {best_epoch} ({best_mae:.3f} m validation MAE).")

## Inference and visualization

Use the best weights to predict depth from a single RGB image. The example compares a held-out image's prediction and LiDAR measurements using the same colour scale. Missing measurements are masked.

In [ ]:
@torch.inference_mode()
def predict_depth(image_path):
    model.eval()
    rgb = read_rgb(image_path)
    prediction = model(rgb.unsqueeze(0).to(device))[0, 0].cpu()
    return rgb, prediction


example_path = val_ds.pairs[0][0]
rgb, prediction = predict_depth(example_path)
_, target = val_ds[0]
target = target[0].numpy()
target = np.ma.masked_less_equal(target, 0)
scale_max = float(np.percentile(target.compressed(), 95)) if target.count() else float(prediction.max())

fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
axes[0].imshow(rgb.permute(1, 2, 0))
axes[0].set_title("RGB")
axes[1].imshow(target, cmap="plasma", vmin=0, vmax=scale_max)
axes[1].set_title("LiDAR ground truth")
plot = axes[2].imshow(prediction, cmap="plasma", vmin=0, vmax=scale_max)
axes[2].set_title("ViT depth")
for axis in axes:
    axis.set_axis_off()
fig.colorbar(plot, ax=axes[1:], label="Depth (m)", extend="max")
plt.show()